# ASG Airlines: End-to-End Data Engineering Case Study

## Problem Statement

ASG Airlines, a nationwide carrier, operates flights across multiple cities through its booking platforms, scheduling systems, and airport logs. The organization collects operational flight data from these various systems; however, inconsistencies in the dataset such as corrupted flight identifiers, inconsistent time formats, missing values, and unhandled overnight (cross-day) flight scenarios have created challenges in generating accurate operational insights.

Without a standardized and reliable data pipeline, the airline faces difficulties in calculating accurate flight durations, analyzing route-wise traffic, identifying delays and anomalies, and making data-driven operational decisions.

To improve operational efficiency and reporting accuracy, ASG Airlines aims to build an end-to-end data engineering pipeline that ingests, cleans, standardizes, and transforms flight data into a reliable analytical dataset for reporting and business intelligence purposes.


In [77]:
import pandas as pd
import hashlib

Loading the dataset: (4 separate dataframes since the file has 4 sheets)

In [ ]:
try:
    flights = pd.read_excel(r"../data/UseCase - Airlines.xlsx", sheet_name = "flights") # flights sheet
    print("Loaded flights:", flights.shape[0], "rows")
except Exception as e:
    print("Failed to load [sheet name]:", e)

Loaded [sheet name]: 1020 rows


In [79]:
flights.head()

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


In [ ]:
try:
    payments = pd.read_excel(r"../data/UseCase - Airlines.xlsx", sheet_name = "payments") # payments sheet
    print("Loaded payments:", payments.shape[0], "rows")
except Exception as e:
    print("Failed to load [sheet name]:", e)

Loaded [sheet name]: 1000 rows


In [81]:
payments.head()

,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


In [82]:
try:
    bookings = pd.read_excel(r"../data/UseCase - Airlines.xlsx", sheet_name = "bookings") # bookings sheet
    print("Loaded [sheet name]:", bookings.shape[0], "rows")
except Exception as e:
    print("Failed to load [sheet name]:", e)

Loaded [sheet name]: 1000 rows


In [83]:
bookings.head()

,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


In [84]:
try:
    passengers = pd.read_excel(r"../data/UseCase - Airlines.xlsx", sheet_name = "passengers") # passengers sheet
    print("Loaded [sheet name]:", passengers.shape[0], "rows")
except Exception as e:
    print("Failed to load [sheet name]:", e)

Loaded [sheet name]: 1039 rows


In [85]:
passengers.head()

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


Starting EDA for the "flights" dataframe first:

In [86]:
flights.shape

(1020, 7)

In [87]:
flights.dtypes

flight_id                    str
airline                      str
source                       str
destination                  str
departure_time    datetime64[us]
arrival_time      datetime64[us]
duration                  object
dtype: object

In [88]:
flights.isna().sum()

flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

In [89]:
flights["flight_id"].duplicated().sum()

np.int64(16)

In [90]:
flights.duplicated().sum()

np.int64(15)

Summary (flights):

- 1020 rows, 7 columns
- "duration" is being treated as object, can be dropped and calculated using "departure_time" and "arrival_time"
- "airline" has 41 null values (does not account for "UNKNOWN" values, which were observed while scanning the dataset file)
- 16 duplicate records

EDA for "payments" dataframe:

In [91]:
payments.shape

(1000, 4)

In [92]:
payments.dtypes

payment_id           str
booking_id           str
amount            object
payment_method       str
dtype: object

In [93]:
payments.isna().sum()

payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64

In [94]:
payments["payment_id"].duplicated().sum()

np.int64(0)

In [95]:
payments.duplicated().sum()

np.int64(0)

Summary (payments):

- 1000 rows, 4 columns
- "amount" is being treated as an object instead of float
- "amount" also has 48 null values 
- No duplicates

EDA for "bookings" dataframe:

In [96]:
bookings.shape

(1000, 9)

In [97]:
bookings.dtypes

booking_id                            str
passenger_id                          str
flight_id                             str
booking_date               datetime64[us]
status                                str
passport_number                       str
seat_number                           str
emergency_contact_name                str
emergency_contact_phone               str
dtype: object

In [98]:
bookings.isna().sum()

booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64

In [99]:
bookings["booking_id"].duplicated().sum()

np.int64(0)

In [100]:
bookings.duplicated().sum()

np.int64(0)

Summary (bookings):

- 1000 rows, 9 columns
- all columns' datatypes are correct
- "status" has 45 null values
- No duplicates

EDA for "passengers" dataframe:

In [101]:
passengers.shape

(1039, 9)

In [102]:
passengers.dtypes

passenger_id                str
first_name                  str
last_name                   str
age                       int64
gender                      str
email                       str
phone                       str
aadhaar_id                int64
date_of_birth    datetime64[us]
dtype: object

In [103]:
passengers.isna().sum()

passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64

In [104]:
passengers["passenger_id"].duplicated().sum()

np.int64(39)

In [105]:
passengers.duplicated().sum()

np.int64(0)

Summary (passengers):

- 1039 rows, 9 columns
- all columns' datatypes are okay, "aadhaar_id" can be treated as string since no calculation is going to be done with it
- "last_name" has 10 null values
- 39 duplicate records

In [106]:
print(flights['flight_id'].duplicated().sum())

16


Handling duplicates (flights):

In [107]:
flight_duplicates = flights[flights["flight_id"].duplicated(keep = False)]
print(flight_duplicates.sort_values("flight_id"))

    flight_id    airline source destination          departure_time  \
253     6F250    UNKNOWN    DEL         BLR 2026-04-20 03:26:41.701   
270     6F250    UNKNOWN    CCU         BLR 2026-04-20 02:23:41.702   
549     AI020        NaN    BOM         BLR 2026-04-19 03:53:41.701   
550     AI020        NaN    BOM         BLR 2026-04-19 03:53:41.701   
141     AI031  Air India    DEL         MAA 2026-04-20 13:05:41.701   
142     AI031  Air India    DEL         MAA 2026-04-20 13:05:41.701   
598     AI043  Air India    CCU         DEL 2026-04-19 00:28:41.701   
599     AI043  Air India    CCU         DEL 2026-04-19 00:28:41.701   
602     AI070  Air India    CCU         DEL 2026-04-19 00:05:41.702   
603     AI070  Air India    CCU         DEL 2026-04-19 00:05:41.702   
83      AI242    UNKNOWN    BLR         CCU 2026-04-20 16:41:41.704   
84      AI242    UNKNOWN    BLR         CCU 2026-04-20 16:41:41.704   
939     SJ037   SpiceJet    BLR         BOM 2026-04-17 19:14:41.701   
940   

In [108]:
flight_duplicates = flights[flights["flight_id"].duplicated(keep = False)].sort_values("flight_id")

for fid, group in flight_duplicates.groupby("flight_id"):
    unique_rows = group.drop(columns = "flight_id").drop_duplicates()
    if len(unique_rows) == 1:
        print(fid, "- Actual duplicate record")
    else:
        print(fid, "- Same flight ID, different flight")

6F250 - Same flight ID, different flight
AI020 - Actual duplicate record
AI031 - Actual duplicate record
AI043 - Actual duplicate record
AI070 - Actual duplicate record
AI242 - Actual duplicate record
SJ037 - Actual duplicate record
SJ118 - Actual duplicate record
SJ142 - Actual duplicate record
SJ146 - Actual duplicate record
UK013 - Actual duplicate record
UK049 - Actual duplicate record
UK139 - Actual duplicate record
UK160 - Actual duplicate record
UK163 - Actual duplicate record
UK180 - Actual duplicate record


As seen earlier in the EDA, "flight_id" had 16 duplicates, but as complete records, there were only 15. So filtering was done to extract those records, group them by their ID and add a marker stating the type of duplicate they were. 

6F250 is repeated twice, but is assigned to different flights. Hence, to differentiate, it will be reassigned as 6F250_B.

In [109]:
index = flights[flights["flight_id"] == "6F250"].index[1]

In [110]:
flights.loc[index, "flight_id"] = "6F250_B"

Handling duplicates (passengers):

In [111]:
passenger_duplicates = passengers[passengers["passenger_id"].duplicated(keep=False)].sort_values("passenger_id")

for pid, group in passenger_duplicates.groupby("passenger_id"):
    unique_rows = group.drop(columns = "passenger_id").drop_duplicates()
    if len(unique_rows) == 1:
        print(pid, "- Actual duplicate record")
    else:
        print(pid, "- Same passenger ID, different details")

P1034 - Same passenger ID, different details
P1102 - Same passenger ID, different details
P1104 - Same passenger ID, different details
P1108 - Same passenger ID, different details
P1140 - Same passenger ID, different details
P1141 - Same passenger ID, different details
P1178 - Same passenger ID, different details
P1216 - Same passenger ID, different details
P1236 - Same passenger ID, different details
P1252 - Same passenger ID, different details
P1331 - Same passenger ID, different details
P1408 - Same passenger ID, different details
P1413 - Same passenger ID, different details
P1469 - Same passenger ID, different details
P1542 - Same passenger ID, different details
P1615 - Same passenger ID, different details
P1630 - Same passenger ID, different details
P1695 - Same passenger ID, different details
P1711 - Same passenger ID, different details
P1727 - Same passenger ID, different details
P1728 - Same passenger ID, different details
P1781 - Same passenger ID, different details
P1807 - Sa

As seen earlier in the EDA, "passengers" had 39 duplicates, based off the IDs alone. And the snippet above suggests that there is no reliable way to determine which PII values are correct. Hence, these records will just be flagged.

In [112]:
passengers["id_flag"] = 'Correct'
passengers.loc[passengers["passenger_id"].isin(passenger_duplicates["passenger_id"].unique()), "id_flag"] = "Duplicate"

Addressing the remaining issues:

In [113]:
payments["amount"] = pd.to_numeric(payments["amount"], errors = "coerce") # payments was earlier being treated as an "object"


In [114]:
payments.dtypes # verifying it

payment_id            str
booking_id            str
amount            float64
payment_method        str
dtype: object

## Moving onto data cleaning:

While observing the "duration" column in the "flights" sheet, one record had the "duration" as "#######". This is because the arrival time mentioned was at an earlier date that the departure time. Hence, the calculation isn't reliable and is to be dropped. It should be recomputed considering the dates' accuracy as well.

In [115]:
difference = flights["arrival_time"] - flights["departure_time"]
print((difference < pd.Timedelta(0)).sum())

1


In [116]:
overnight_mask = flights["arrival_time"] < flights["departure_time"]
flights.loc[overnight_mask, "arrival_time"] = flights.loc[overnight_mask,"arrival_time"] + pd.DateOffset(days=1)

flights["duration"] = flights["arrival_time"] - flights["departure_time"]

In [117]:
print((flights['duration'] < pd.Timedelta(0)).sum())

0


The "duration" column values should be accurate now.

We don't require the 15 duplicate (row - based) records too. We can safely delete those.

In [118]:
flights = flights.drop_duplicates()

In [119]:
flights.shape

(1005, 7)

There were initially 1020 records, and so 1020 - 15 = 1005, so the count is correct.

Now, for the "airlines" column, there were 41 nulls, but while scanning through the dataset, there were "UNKNOWN" values as well. This can easily be fixed, since every airline has their unique brand ID.

In [120]:
flights["airline"].value_counts()

airline
IndiGo       249
SpiceJet     236
Air India    233
Vistara      218
UNKNOWN       30
Name: count, dtype: int64

In [121]:
clean = flights[~flights["airline"].isin(["UNKNOWN"]) & flights["airline"].notna()]
prefix_map = clean.groupby(clean["flight_id"].str.extract(r'^([A-Z0-9]+?)\d')[0])["airline"].unique()
print(prefix_map)

0
6F       [IndiGo]
AI    [Air India]
SJ     [SpiceJet]
UK      [Vistara]
Name: airline, dtype: object


In [122]:
lookup = clean.assign(prefix = clean["flight_id"].str.extract(r'^([A-Z0-9]+?)\d')[0]) \
              .groupby("prefix")["airline"].first().to_dict()
print(lookup)

{'6F': 'IndiGo', 'AI': 'Air India', 'SJ': 'SpiceJet', 'UK': 'Vistara'}


In [123]:
missing_mask = flights["airline"].isna() | (flights["airline"] == "UNKNOWN")
flights.loc[missing_mask, "airline"] = flights.loc[missing_mask, "flight_id"].str.extract(r'^([A-Z0-9]+?)\d')[0].map(lookup)

In [124]:
print(flights["airline"].isna().sum())
print((flights["airline"] == "UNKNOWN").sum())

0
0


So now, the "airline" column should have no nulls/"UNKNOWN" values based of their flight numbers.

In [125]:
# quick check

print(flights.shape)
print(flights.isna().sum())

(1005, 7)
flight_id         0
airline           0
source            0
destination       0
departure_time    0
arrival_time      0
duration          0
dtype: int64


"flights" now has 1005 rows and 7 columns, with no nulls and accurate datatypes.

Now, the Personally Identifiable Information (PII) from "passengers" sheet needs to be masked.

In [126]:
def hash_value(val):
    if pd.isna(val):
        return val
    return hashlib.sha256(str(val).encode()).hexdigest()

passengers["aadhaar_id"] = passengers["aadhaar_id"].apply(hash_value)

In [127]:
passengers[["passenger_id", "aadhaar_id"]].head() # verifying it worked

,passenger_id,aadhaar_id
0,P1000,99466baa9b8fe69a6f31db34c87d50f0e9a488eda788f3...
1,P1001,6d05ccbeb5011dd59948b882fff86e1a6f7fc0cf41f659...
2,P1002,3e24ac79e2c956797c22051676644d26204cc4de0f203e...
3,P1003,4e880723014ee24b39f0213150496f3efb9faa3f62d335...
4,P1004,3ca09c31f18da71ad2bc39cb3cbb387c38a9cd5ad37239...


This hashing also needs to be applied to the "passport_number", "emergency_contact_name" and "emergency_contact_phone" columns from the "bookings" sheet.

In [128]:
bookings["passport_number"] = bookings["passport_number"].apply(hash_value)
bookings["emergency_contact_name"] = bookings["emergency_contact_name"].apply(hash_value)
bookings["emergency_contact_phone"] = bookings["emergency_contact_phone"].apply(hash_value)

"email" and "phone" columns from the "passengers" sheet can be masked.

In [129]:
def mask_email(val):
    if pd.isna(val):
        return val
    name, domain = val.split("@")
    return name[0] + "***@" + domain

passengers["email"] = passengers["email"].apply(mask_email)

In [130]:
passengers["email"].head()

0      v***@gmail.com
1    k***@hotmail.com
2    m***@outlook.com
3    m***@hotmail.com
4    s***@outlook.com
Name: email, dtype: str

In [131]:
def mask_phone(val):
    if pd.isna(val):
        return val
    return val[:6] + "XXXX" + val[-2:]

passengers["phone"] = passengers["phone"].apply(mask_phone)

In [132]:
passengers["phone"].head()

0    +91-68XXXX90
1    +91-67XXXX97
2    +91-61XXXX92
3    +91-87XXXX51
4    +91-78XXXX13
Name: phone, dtype: str

Passengers' DOB can be generalized as well.

In [133]:
passengers["date_of_birth"] = passengers["date_of_birth"].dt.year

In [134]:
passengers["date_of_birth"].head()

0    1974
1    2011
2    1954
3    1965
4    2005
Name: date_of_birth, dtype: int32

In [135]:
route_stats = flights.groupby(["source", "destination"])["duration"].transform("mean")
route_std = flights.groupby(["source", "destination"])["duration"].transform("std")

In [136]:
flights["is_anomaly"] = (flights["duration"] > flights["duration"].quantile(0.95)) | (flights["duration"] < flights["duration"].quantile(0.05))

In [137]:
flights["is_anomaly"].sum()

np.int64(94)

In [138]:
flights["duration_minutes"] = flights["duration"].dt.total_seconds() / 60 # for PowerBI

In [139]:
flights.dtypes

flight_id                       str
airline                         str
source                          str
destination                     str
departure_time       datetime64[us]
arrival_time         datetime64[us]
duration            timedelta64[us]
is_anomaly                     bool
duration_minutes            float64
dtype: object

In [140]:
payments.shape

(1000, 4)

In [141]:
payments.dtypes

payment_id            str
booking_id            str
amount            float64
payment_method        str
dtype: object

In [142]:
bookings.shape

(1000, 9)

In [143]:
bookings.dtypes

booking_id                            str
passenger_id                          str
flight_id                             str
booking_date               datetime64[us]
status                                str
passport_number                       str
seat_number                           str
emergency_contact_name                str
emergency_contact_phone               str
dtype: object

In [144]:
passengers.shape

(1039, 10)

In [145]:
passengers.dtypes

passenger_id       str
first_name         str
last_name          str
age              int64
gender             str
email              str
phone              str
aadhaar_id         str
date_of_birth    int32
id_flag            str
dtype: object

In [146]:
flights.to_csv("../data/cleaned/flights_cleaned.csv", index = False)
bookings.to_csv("../data/cleaned/bookings_cleaned.csv", index = False)
passengers.to_csv("../data/cleaned/passengers_cleaned.csv", index = False)